# Depth-cued change blindness: analysis

Measures from `*_trials.csv` files:

- **alternations to identification**: `rt_ms / alternation_s`, the number of A↔A' switches seen before SPACE (Rensink, O'Regan & Clark, 1997, p. 370), on **correct** trials only (place and kind of change both right);
- by **plane of the change** (near, far) and **cue** (valid, invalid, neutral);
- the **cue effect**: invalid − valid alternations, per plane, per subject; with neutral trials, the benefit (neutral − valid) and the cost (invalid − neutral);
- identification errors and missed changes per cell;
- a check that the changed object's eccentricity and changed area do not differ between cue conditions.

`ANALYSIS_RUN` (or `RUN` below) is a run folder or any folder above several runs. Subject means are computed first; tests across subjects are exact sign-flip permutation tests.

On simulated data the numbers are the simulator's invented ones (src/depthcb/simulate.py); the notebook then only shows that the analysis runs.

In [ ]:
import os
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use('Agg')
import matplotlib.pyplot as plt

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
RUN = os.environ.get('ANALYSIS_RUN')
if RUN is None:
    runs = sorted(HERE.glob('data*/**/*_trials.csv'), key=lambda p: p.stat().st_mtime)
    RUN = runs[-1].parent if runs else None
print('reading under:', RUN)

In [ ]:
files = sorted(Path(RUN).rglob('*_trials.csv')) if RUN else []
frames = []
for f in files:
    d = pd.read_csv(f)
    d['subject'] = f.name.split('_')[0]
    frames.append(d)
trials = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(len(files), 'file(s),', len(trials), 'trials,', trials['subject'].nunique() if len(trials) else 0, 'subject(s)')
if len(trials):
    print(pd.crosstab([trials['plane'], trials['cue']], trials['outcome']).to_string())

## Errors and missed changes

In [ ]:
if len(trials):
    t = trials.copy()
    t['missed'] = t['outcome'] == 'NO_DETECTION'
    t['identified'] = t['outcome'].isin(['CORRECT', 'MISIDENTIFIED'])
    t['error'] = t['outcome'] == 'MISIDENTIFIED'
    summary = t.groupby(['plane', 'cue']).agg(trials=('outcome', 'size'),
                                             missed_pct=('missed', 'mean'),
                                             error_pct=('error', 'mean'))
    summary[['missed_pct', 'error_pct']] *= 100
    print(summary.round(1).to_string())

## Alternations to identification by plane and cue (correct trials)

In [ ]:
correct = trials[trials['outcome'] == 'CORRECT'].copy() if len(trials) else pd.DataFrame()
CUE_ORDER = [c for c in ['valid', 'neutral', 'none', 'invalid'] if len(correct) and c in set(correct['cue'])]
if len(correct):
    per_subject = (correct.groupby(['subject', 'plane', 'cue'])['alternations']
                   .mean().rename('alternations').reset_index())
    table = per_subject.groupby(['plane', 'cue'])['alternations'].agg(['mean', 'sem', 'count']).round(2)
    print('mean alternations per subject, then across subjects (count = subjects):')
    print(table.to_string())
    print()
    print('trial medians (a check against skew):')
    print(correct.groupby(['plane', 'cue'])['alternations'].median().round(2).to_string())

## Cue effect: invalid − valid, per plane

Positive means the depth cue helped. With two or more subjects the paired subject differences get an exact sign-flip test; with one subject there is nothing to test across observers.

In [ ]:
def sign_flip_p(diffs):
    diffs = np.asarray(diffs, float)
    n = len(diffs)
    if n == 0:
        return float('nan')
    observed = abs(diffs.mean())
    signs = np.array(np.meshgrid(*[[-1, 1]] * n)).reshape(n, -1).T if n <= 16 else \
        np.random.default_rng(0).choice([-1, 1], size=(20000, n))
    return float((np.abs((signs * diffs).mean(axis=1)) >= observed - 1e-12).mean())

if len(correct):
    wide = per_subject.pivot_table(index=['subject', 'plane'], columns='cue', values='alternations').reset_index()
    rows = []
    for plane, g in wide.groupby('plane'):
        for name, hi, lo in [('cue effect (invalid - valid)', 'invalid', 'valid'),
                             ('benefit (neutral - valid)', 'neutral', 'valid'),
                             ('cost (invalid - neutral)', 'invalid', 'neutral')]:
            if {hi, lo} <= set(g.columns):
                d = (g[hi] - g[lo]).dropna()
                rows.append({'plane': plane, 'contrast': name, 'mean': round(d.mean(), 2),
                             'subjects': len(d), 'sign-flip p': round(sign_flip_p(d), 4) if len(d) >= 2 else None})
    print(pd.DataFrame(rows).to_string(index=False))
    if {'invalid', 'valid'} <= set(wide.columns):
        both = wide.assign(effect=wide['invalid'] - wide['valid']).pivot_table(index='subject', columns='plane', values='effect').dropna()
        if len(both) >= 2 and {'near', 'far'} <= set(both.columns):
            d = both['near'] - both['far']
            print(f'cue effect near - far: {d.mean():.2f} alternations, n = {len(d)}, sign-flip p = {sign_flip_p(d):.4f}')

## Balance check: the changed object across cue conditions

The design draws the changed object within the same eccentricity band on both planes and assigns cues within each plane × type cell, so eccentricity and changed area should not differ between valid and invalid trials. Changed area does differ between planes when relative size is on (near objects are larger and cast shadows); the Size-matched parameter set removes the size difference.

In [ ]:
if len(trials):
    print(trials.groupby(['plane', 'cue'])[['eccentricity_dva', 'changed_area_dva2']].mean().round(2).to_string())

## Plot

In [ ]:
if len(correct):
    fig, ax = plt.subplots(figsize=(5.6, 3.6), dpi=110)
    colours = {'valid': '#3a6ea5', 'neutral': '#9a9a9a', 'none': '#cfcfcf', 'invalid': '#b5523b'}
    width = 0.8 / max(len(CUE_ORDER), 1)
    for j, cue in enumerate(CUE_ORDER):
        means = [table['mean'].get((pl, cue), np.nan) for pl in ['near', 'far']]
        sems = [table['sem'].get((pl, cue), np.nan) for pl in ['near', 'far']]
        ax.bar(np.arange(2) + (j - (len(CUE_ORDER) - 1) / 2) * width, means, width * 0.95,
               yerr=np.nan_to_num(sems), color=colours[cue], label=cue)
    ax.set_xticks(range(2), ['near-plane change', 'far-plane change'])
    ax.set_ylabel('alternations to identification')
    ax.legend(frameon=False, fontsize=8, title='depth cue', title_fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    out = Path(RUN) / 'alternations_by_plane_and_cue.png'
    fig.savefig(out)
    print('saved', out)